<a href="https://colab.research.google.com/github/mdmostafizurrahman6351/Deep_Learning_Project/blob/main/project_04_image_classification_for_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D


import matplotlib.pyplot as plt
import pickle

from sklearn.metrics import accuracy_score

Dataset load

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(x_train.shape)
print(x_test.shape)

In [ ]:
plt.imshow(x_train[0])


Normalization

In [ ]:
x_train = x_train / 255
x_test = x_test / 255

x_test[0]

Building ANN

In [ ]:
model = Sequential()

model.add(Flatten(input_shape=(28, 28)))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(10, activation='softmax'))



model.summary()

In [ ]:
# model compile
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
history = model.fit(x_train, y_train, epochs=15, validation_split=0.1)

In [ ]:
# evoulate
y_pred = model.predict(x_test)

In [ ]:
accuracy_score(y_test, y_pred.argmax(axis=1))

In [ ]:
plt.figure(figsize=(15, 6)) # Adjust figure size to better fit 10 images

for i in range(10):
    plt.subplot(2, 5, i + 1) # Create a 2x5 grid of subplots
    plt.imshow(x_test[i], cmap=plt.cm.binary) # Display image in grayscale
    actual_label = y_test[i]
    predicted_label = y_pred.argmax(axis=1)[i]
    plt.title(f"Actual: {actual_label}\nPred: {predicted_label}")
    plt.axis('off') # Hide axes for cleaner image display

plt.tight_layout() # Adjust layout to prevent titles/labels from overlapping
plt.show()

Predction system


## Prediction System

This section will implement a system to predict digits from an image URL. It will involve loading the image, preprocessing it to match the training data (28x28 grayscale), and then using our trained model for prediction.

In [ ]:
import requests
from PIL import Image
import numpy as np
from io import BytesIO
import matplotlib.pyplot as plt

def predict_digit_from_url(image_url):
    try:
        # Fetch the image from the URL
        response = requests.get(image_url)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        img = Image.open(BytesIO(response.content))

        # --- Display original image ---
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
        plt.imshow(img)
        plt.title("Original Image")
        plt.axis('off')
        # --------------------------------

        # Convert to grayscale and resize to 28x28
        img_gray = img.convert('L')
        img_resized = img_gray.resize((28, 28))

        # Convert to numpy array and normalize
        img_array = np.array(img_resized)
        img_array = img_array / 255.0  # Normalize to 0-1

        # Invert colors if the digit appears white on a dark background
        if np.mean(img_array) > 0.5: # Arbitrary threshold, can be adjusted
            img_array = 1.0 - img_array # Invert colors (0 becomes 1, 1 becomes 0)
            print("Image colors inverted for better prediction.")

        # Reshape for the model (add batch dimension)
        img_for_prediction = img_array.reshape(1, 28, 28)

        # --- Display preprocessed image ---
        plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
        plt.imshow(img_array, cmap=plt.cm.binary)
        plt.title("Processed Image for Prediction")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        # -------------------------------------

        # Make prediction
        prediction = model.predict(img_for_prediction)
        predicted_digit = np.argmax(prediction)

        return predicted_digit

    except requests.exceptions.RequestException as e:
        print(f"Error fetching image from URL: {e}")
        return None
    except Exception as e:
        print(f"An error occurred during image processing or prediction: {e}")
        return None

Now, let's test the prediction system with a sample image URL. You can replace the `sample_image_url` with any image URL of a handwritten digit (preferably a clear, black and white image for best results, similar to MNIST data).

In [ ]:
# Example usage:
# You can find sample handwritten digit images online or create your own.
# For best results, use a clear, single digit image with a white background and dark digit.
sample_image_url = 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT5wEM43b_-p0ljmSrH5u5QfIqt_an6X8JeoGbdFVagug&s' # Trying a publicly accessible image of digit '2' from PyTorch examples on GitHub

predicted_result = predict_digit_from_url(sample_image_url)

if predicted_result is not None:
    print(f"Predicted Digit: {predicted_result}")